# Day 10 Project Solution: Templated Prompt Engine

**Deliverable:** Loads a JSON registry of named, versioned prompt templates (`summarize_v1`, `classify_v1`, `extract_v1`), renders each with sample variables, and calls the local Ollama model. Output shows three separate template-driven LLM interactions, each constructed without a single hard-coded f-string in the calling code.

## Setup

In [ ]:
import json
import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from string import Template

import ollama

# Lesson 5: log the render step so every prompt is inspectable in the output
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(message)s",
)
logger = logging.getLogger(__name__)

MODEL = "llama3.2"
REGISTRY_PATH = Path("prompt_registry.json")

## Lesson 3 — PromptTemplate Dataclass

Four fields make a bare `string.Template` self-documenting and validatable.

In [ ]:
@dataclass
class PromptTemplate:
    """A self-documenting, validatable prompt template (Lesson 3)."""

    name: str
    template_str: str
    required_vars: list  # placeholder names the caller must supply
    description: str

    def validate(self, values: dict) -> None:
        """Raise ValueError listing every required variable absent from values."""
        # Collect all missing vars in one pass — better UX than KeyError on first miss
        missing = [v for v in self.required_vars if v not in values]
        if missing:
            raise ValueError(
                f"PromptTemplate '{self.name}' is missing required "
                f"variable(s): {missing}"
            )

    def render(self, values: dict) -> str:
        """Validate values, then return the fully substituted prompt string."""
        self.validate(values)  # fail loudly before touching string.Template
        return Template(self.template_str).substitute(values)


# Quick unit test — no model needed (Lesson 1: pure template, no side effects)
_test_pt = PromptTemplate(
    name="_test",
    template_str="Hello $name!",
    required_vars=["name"],
    description="Test template.",
)
assert _test_pt.render({"name": "world"}) == "Hello world!"
try:
    _test_pt.validate({})
    assert False, "should have raised"
except ValueError as e:
    assert "name" in str(e)
print("PromptTemplate: validate and render work correctly")

## Lesson 4 — Registry with Immutability and JSON Persistence

In [ ]:
# Plain dict: key = 'name_vN', value = PromptTemplate
_registry: dict = {}


def register(pt: PromptTemplate) -> PromptTemplate:
    """Add a PromptTemplate to the registry. Raises ValueError if key exists.

    Immutability rule: never overwrite an existing version — add a new one.
    """
    if pt.name in _registry:
        raise ValueError(
            f"Template '{pt.name}' already exists. "
            "Increment the version number instead of mutating an existing entry."
        )
    _registry[pt.name] = pt
    return pt  # return so register() can wrap a definition inline


def get_template(key: str) -> PromptTemplate:
    """Look up a PromptTemplate by key. Raises KeyError if not found."""
    return _registry[key]  # let KeyError propagate — fail loudly


def list_versions(name: str) -> list:
    """Return all registered keys whose name prefix matches, sorted."""
    prefix = f"{name}_v"
    return sorted(k for k in _registry if k.startswith(prefix))


def save_registry(path) -> None:
    """Write the full registry to a JSON file for persistence and auditing."""
    data = {
        key: {
            "template_str": pt.template_str,
            "required_vars": pt.required_vars,
            "description": pt.description,
        }
        for key, pt in _registry.items()
    }
    with open(path, "w") as fh:
        json.dump(data, fh, indent=2)


def load_registry(path) -> None:
    """Load a saved registry JSON file, merging entries into the live registry.

    Calls register() for each entry, so immutability is enforced during load.
    """
    with open(path) as fh:
        data = json.load(fh)
    for key, entry in data.items():
        pt = PromptTemplate(
            name=key,
            template_str=entry["template_str"],
            required_vars=entry["required_vars"],
            description=entry["description"],
        )
        register(pt)


print("Registry functions defined.")

## Lesson 2 — Define Three Templates Using string.Template Syntax

`$variable` placeholders, no brace-doubling, safe inside JSON and code blocks.

In [ ]:
# All three templates use $placeholder syntax — no f-strings anywhere in the text.

SUMMARIZE = register(PromptTemplate(
    name="summarize_v1",
    description="Summarise a text passage in a fixed number of sentences.",
    required_vars=["text", "max_sentences"],
    template_str=(
        "Summarise the following text in exactly $max_sentences sentence(s).\n"
        "Be concise and factual. Do not add information not present in the text.\n\n"
        "$text"
    ),
))

CLASSIFY = register(PromptTemplate(
    name="classify_v1",
    description="Classify text into exactly one of the given categories.",
    required_vars=["text", "categories"],
    template_str=(
        "Classify the text below as exactly one of: $categories.\n"
        "Reply with the category label only — no punctuation, no explanation.\n\n"
        "$text"
    ),
))

# Braced form ${entity_type}s — needed because the placeholder adjoins the letter 's'
EXTRACT = register(PromptTemplate(
    name="extract_v1",
    description="Extract all instances of a named entity type as a JSON array.",
    required_vars=["entity_type", "text"],
    template_str=(
        "Extract all ${entity_type}s from the text below.\n"
        "Return a JSON array of strings — no other text.\n\n"
        "$text"
    ),
))

print("Registered templates:", list(_registry.keys()))

## Lesson 4 — Save and Reload the Registry

In [ ]:
# Persist to disk — the JSON file is both a persistence layer and an audit log
save_registry(REGISTRY_PATH)
print(f"Registry saved to: {REGISTRY_PATH.resolve()}")
print(f"File size: {REGISTRY_PATH.stat().st_size} bytes")

# Demonstrate immutability: attempting to overwrite an existing key raises ValueError
try:
    register(PromptTemplate(
        name="summarize_v1",  # already registered above
        template_str="A completely different prompt.",
        required_vars=["text"],
        description="Should not be allowed.",
    ))
    print("ERROR: should have raised")
except ValueError as e:
    print(f"Immutability enforced: {e}")

# Round-trip verification: clear, reload, check keys survived
_registry.clear()
load_registry(REGISTRY_PATH)
assert set(_registry.keys()) == {"summarize_v1", "classify_v1", "extract_v1"}
print("Round-trip load verified. Keys:", list(_registry.keys()))

## Lesson 5 — render() and render_and_call()

In [ ]:
def render(key: str, variables: dict) -> str:
    """Fill a registry template with the given variables.

    This is the pure construction step — no model call, fully testable.

    Args:
        key:       Registry key in 'name_vN' format.
        variables: Mapping of placeholder names to string values.

    Returns:
        The fully filled prompt string.

    Raises:
        KeyError:   If key is not in the registry.
        ValueError: If a required placeholder is missing from variables.
    """
    pt = get_template(key)  # KeyError if key absent
    filled = pt.render(variables)  # ValueError if a required var is missing
    logger.debug("render(%r) → %d chars", key, len(filled))
    return filled


def render_and_call(
    key: str,
    variables: dict,
    model: str = MODEL,
    system_prompt: str = None,
) -> str:
    """Render a template, call the local Ollama model, return its reply.

    Args:
        key:           Registry key for the template to render.
        variables:     Placeholder values to fill into the template.
        model:         Ollama model name.
        system_prompt: Optional system message prepended before the user turn.

    Returns:
        The model's reply as a plain string.
    """
    # Step 1 — render: template + variables → plain string (the seam you can test)
    user_message = render(key, variables)

    # Step 2 — build: construct the messages list (same format as Day 3 onwards)
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})

    logger.info("render_and_call: model=%s template=%r", model, key)

    # Step 3 — call: filled string → model response
    response = ollama.chat(model=model, messages=messages)
    reply = response["message"]["content"]

    logger.debug("reply: %d chars", len(reply))
    return reply


# Unit tests for the render step — no Ollama required
filled = render("summarize_v1", {"text": "RAG combines retrieval with generation.", "max_sentences": "1"})
assert "RAG" in filled and "1 sentence" in filled
print("render() unit test passed (no model call)")

try:
    render("no_such_key", {})
    assert False, "should have raised"
except KeyError:
    print("render() raises KeyError for unknown key — correct")

## Three Template-Driven LLM Interactions

All three calls go through `render_and_call()`. No f-strings in the calling code — every prompt lives in the registry.

In [ ]:
SAMPLE_TEXT = (
    "The Python programming language was created by Guido van Rossum "
    "and first released in 1991. It emphasises code readability and "
    "supports multiple programming paradigms including procedural, "
    "object-oriented, and functional programming."
)

separator = "-" * 60

# ── Call 1: summarize_v1 ─────────────────────────────────────────────────────
print(separator)
print("TEMPLATE: summarize_v1")
print("RENDERED PROMPT:")
print(render("summarize_v1", {"text": SAMPLE_TEXT, "max_sentences": "2"}))
print()

summary = render_and_call(
    "summarize_v1",
    {"text": SAMPLE_TEXT, "max_sentences": "2"},
)
print("MODEL RESPONSE:")
print(summary)

In [ ]:
# ── Call 2: classify_v1 ─────────────────────────────────────────────────────
print(separator)
print("TEMPLATE: classify_v1")
print("RENDERED PROMPT:")
print(render("classify_v1", {"text": SAMPLE_TEXT, "categories": "history, technology, science, art"}))
print()

label = render_and_call(
    "classify_v1",
    {"text": SAMPLE_TEXT, "categories": "history, technology, science, art"},
    system_prompt="You are a concise text classifier. Reply with the category label only.",
)
print("MODEL RESPONSE:")
print(label.strip())

In [ ]:
# ── Call 3: extract_v1 ──────────────────────────────────────────────────────
print(separator)
print("TEMPLATE: extract_v1")
print("RENDERED PROMPT:")
print(render("extract_v1", {"entity_type": "year", "text": SAMPLE_TEXT}))
print()

extracted = render_and_call(
    "extract_v1",
    {"entity_type": "year", "text": SAMPLE_TEXT},
)
print("MODEL RESPONSE:")
print(extracted.strip())

## Deliverable Confirmation

In [ ]:
# Final gate check — verify Ollama is reachable (requests was introduced in Day 3)
import requests
try:
    requests.get("http://localhost:11434/api/tags", timeout=3).raise_for_status()
except Exception as e:
    raise AssertionError(f"Ollama server is not running: {e}") from e

# Confirm deliverable conditions
assert REGISTRY_PATH.exists(), "Registry JSON file was not saved"
assert len(_registry) == 3, f"Expected 3 templates, found {len(_registry)}"
assert "summarize_v1" in _registry
assert "classify_v1" in _registry
assert "extract_v1" in _registry

print(separator)
print("Day 10 deliverable produced.")
print(f"  Registry:  {len(_registry)} versioned templates saved to {REGISTRY_PATH}")
print("  Templates: summarize_v1, classify_v1, extract_v1")
print("  Calls:     3 template-driven LLM interactions completed")
print("  F-strings: 0 in calling code — all prompt text lives in the registry")
print(separator)